# Figures for THP Annual Progress Report

In [ ]:
import sys
import contextily as ctx

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

import geopandas as gpd
from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
# Selected date for plotting
dt = '20231210'

In [ ]:
# HRRR met forcings over Animas River Basin?
# Read in .grib2 files from HRRR model output and plot one hour of DSWRF, DLWRF and TMP for december 10, 2024
hrrr_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/patchwork_HRRR/hrrr.{dt}'
hh = '18'
hrrr_fn = h.fn_list(hrrr_dir, f'*t{hh}z.*f01.grib2')[0]
print(hrrr_fn)

# Load the grib file with xarray
dropvarlist = ['orog', 'tp', 'vbdsf', 'vddsf']
ds = xr.open_dataset(hrrr_fn, engine='cfgrib', filter_by_keys={'typeOfLevel': 'surface'}, drop_variables=dropvarlist)
# ds

# Find extent for this grib file
lon_min, lon_max = ds.longitude.min().item(), ds.longitude.max().item()
lat_min, lat_max = ds.latitude.min().item(), ds.latitude.max().item()
print(f'Longitude extent: {lon_min-360} to {lon_max-360}')
print(f'Latitude extent: {lat_min} to {lat_max}')
# Print the span of the data
print(f'Longitude span: {lon_max - lon_min}')
print(f'Latitude span: {lat_max - lat_min}')

# Plot DSWRF and DLWRF and add basemap for context
# Make sure the datasets are plotted such that "x" is latitude and plotted on the y axis
# and "y" is longitude and plotted on the x axis
fig, axa = plt.subplots(1, 3, figsize=(14, 4))
vmin, vmax = 0, 600
ds.dswrf.plot(ax=axa[0], x='longitude', y='latitude', cmap='inferno', vmin=vmin, vmax=vmax, cbar_kwargs={'fraction':0.046, 'pad':0.04})
axa[0].set_title('Downward Shortwave Radiation Flux (W/m^2)')
ds.dlwrf.plot(ax=axa[1], x='longitude', y='latitude', cmap='inferno', vmin=vmin, vmax=vmax, cbar_kwargs={'fraction':0.046, 'pad':0.04})
axa[1].set_title('Downward Longwave Radiation Flux (W/m^2)')

# Use contextily to add a basemap over this area
axa[2].set_xlim(lon_min-360, lon_max-360)
axa[2].set_ylim(lat_min, lat_max)
ctx.add_basemap(axa[2], crs='EPSG:4326')
for ax in axa:
    h.clean_axes(ax)
plt.suptitle(f'HRRR Model Output at {dt} {hh}Z')
plt.tight_layout()

In [ ]:
# Plot a downscaled katana wind output
basin = 'animas'
wind_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/katana_basins/{basin}_100m_katana/data{dt}/wind_ninja_data/'

# Read in an hour of wind velocity data from this directory
# First convert the ascii to tif using gdal_translate if not already done
# gdal_translate -of GTiff topo_windninja_topo_12-10-2023_1800_200m_vel.asc topo_windninja_topo_12-10-2023_1800_200m_vel.tif
wind_fn = h.fn_list(wind_dir, f'*_{hh}*vel.tif')[0]
print(wind_fn)

# Load the wind ascii file, skipping the first 6 rows of header information (suggested for .asc files)
# ncols	248
# nrows	412
# xllcorner	229889.296875
# yllcorner	4123258.000000
# cellsize	200.000000
# NODATA_value	-9999.000000
# wind_file = np.loadtxt(wind_fn, skiprows=6)


# Load the wind tif file using xarray
wind_ds = np.squeeze(xr.open_dataset(wind_fn, engine='rasterio'))

# Rename the data variable to something more manageable
wind_ds = wind_ds.rename({'band_data': 'Wind Velocity (m/s)'})

# Add some unit metadata
wind_ds = wind_ds.assign_attrs({'units': 'm/s', 'long_name': 'Wind Velocity'})

# Plot the wind velocity data
fig, axa = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
wind_ds['Wind Velocity (m/s)'].plot(ax=axa[0], cmap='BuPu', cbar_kwargs={'fraction':0.046, 'pad':0.04})
h.clean_axes(axa[0], labelsoff=True)

# Add basemap to the same extent as the wind data
ctx.add_basemap(axa[1], crs='EPSG:32613')
plt.suptitle(f'{basin.capitalize()} Basin Downscaled Wind Velocity at {dt} {hh}Z')

# Read in the polygons for the basin boundary and plot on top of the basemap
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
basin_poly_fn = h.fn_list(script_dir, f'{basin}_setup/polys/{basin}*.shp')[0]
print(basin_poly_fn)

# Load the basin polygon using geopandas
basin_gdf = gpd.read_file(basin_poly_fn)
basin_gdf.to_crs(epsg=32613).plot(ax=axa[0], facecolor='none', edgecolor='k', linewidth=1.5)
basin_gdf.to_crs(epsg=32613).plot(ax=axa[1], facecolor='none', edgecolor='k', linewidth=1.5)

plt.tight_layout()

In [ ]:
# SPIReS snow albedo over study area/WUS
spires_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/SPIReS_albedo/'
spires_fn = h.fn_list(spires_dir, f'*/*{dt}_albedo.tif')[0]
print(spires_fn)

# Load the SPIReS albedo GeoTIFF with xarray
ds_alb = np.squeeze(xr.open_dataset(spires_fn))

# Rename "band_data" to "albedo"
ds_alb = ds_alb.rename({'band_data': 'albedo'})

# Reassign no data values () to nan
ds_alb = ds_alb.where(ds_alb.albedo != 0)

# Reproject ds_alb to EPSG:4326
ds_alb = ds_alb.rio.reproject('EPSG:4326')
# ds_alb['albedo'].plot.imshow()

# Plot it along with a basemap
fig, axa = plt.subplots(1, 2, figsize=(12, 3), sharex=True, sharey=True)
ds_alb['albedo'].plot.imshow(ax=axa[0], cmap='viridis', vmin=0, vmax=10000, cbar_kwargs={'fraction':0.046, 'pad':0.04})
axa[0].set_title('SPIReS Snow Albedo (x10000)')

# Use contextily to add a basemap over this area
lats = ds_alb.y.min().item(), ds_alb.y.max().item()
lons = ds_alb.x.min().item(), ds_alb.x.max().item()
print(f'Albedo extent lons: {lons}, lats: {lats}')

# Get the crs of the albedo data
crs_alb = ds_alb.rio.crs
print(crs_alb)
ctx.add_basemap(axa[1], crs=crs_alb)
for ax in axa:
    h.clean_axes(ax)
    ax.set_xlim(-125, -104)
    ax.set_ylim(34, 48)

plt.suptitle(f'SPIReS Snow Albedo at {dt}')

In [ ]:
# Plot it on top of a basemap
fig, ax = plt.subplots(1, 1, figsize=(9, 5))

# Plot the albedo data on top of the basemap (do not use imshow here)
ds_alb['albedo'].plot(cmap='magma', vmin=0, vmax=10000)
ctx.add_basemap(ax, crs=crs_alb, zoom=7)

ax.set_xlim(-125, -104)
ax.set_ylim(34, 48)

# Get the crs of the albedo data
h.clean_axes(ax)
ax.set_title(f'SPIReS MODIS Snow Albedo on {dt}')

In [ ]:
w, s, e, n = -125, 34, -104, 48
ctx.tile._calculate_zoom(w, s, e, n)

In [ ]:
ctx.howmany(w, s, e, n, 7, ll=True)

In [ ]:
# SWI maps for one date over -3- water years in Animas River Basin
basin_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/{basin}_100m_isnobal'
# From the selected date, pull the MMDD
# mmdd = dt[4:8]
mmdd = '0411'
print(mmdd)

swi_ds = xr.open_mfdataset(h.fn_list(basin_dir, f'wy*/*unified/*{mmdd}/em.nc'), combine='by_coords')
swi_ds

In [ ]:
# Plot the 2300 hour accumulation (19-23) for each water year
fig, axa = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

# Extract the unique years from the time dimension
for i, year in enumerate(np.unique(swi_ds['time'].dt.year.values)):
    print(year)
    # Plot the SWI for 2300 hour of that year
    swi_ds['SWI'].sel(time=np.datetime64(f'{year}-{mmdd[:2]}-{mmdd[2:]}T00:00:00')).plot(ax=axa[i], cmap='Blues', vmin=0, vmax=3, cbar_kwargs={'fraction':0.046, 'pad':0.04})
    # swi_ds.swi.sel(time='23:00').sel().plot(ax=axa[i], cmap='Blues', vmin=0, vmax=500, cbar_kwargs={'fraction':0.046, 'pad':0.04})
    # axa[i].set_title(f'Water Year {wy} SWI on {mmdd}')
    h.clean_axes(axa[i])


In [ ]:
# snow depth vs. SNOTEL for Animas (all years)

In [ ]:
# EB plots for Animas (melt contributions from SW, LW, H, LE) for all available years